In [1]:
# Cell 1 — Imports
import os
import torch
import torch.nn as nn
import mlflow
import mlflow.pytorch
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, random_split
from pathlib import Path

print(f"PyTorch: {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")

PyTorch: 2.12.0
MPS available: True


In [ ]:
load_dotenv(Path("../.env"))
mlflow.set_tracking_uri(os.getenv("MLFLOW_URI"))
mlflow.set_experiment("fish-classifier-expanded")

In [2]:
# Cell 2 — Config
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
DATA_DIR = Path("/Volumes/kaggle/inat_fish")
MODEL_SAVE_PATH = Path("../models/efficientnet_b4_fish_expanded.pth")
BATCH_SIZE = 32
EPOCHS = 15
LR = 0.001
IMG_SIZE = 380  # B4 native resolution
NUM_CLASSES = len(list(DATA_DIR.iterdir()))

print(f"Device: {DEVICE}")
print(f"Classes found: {NUM_CLASSES}")

FileNotFoundError: [Errno 2] No such file or directory: '/Volumes/kaggle/inat_fish'

In [ ]:
# Cell 3 — Transforms
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
# Cell 4 — Dataset
full_dataset = datasets.ImageFolder(DATA_DIR, transform=train_transforms)
classes = full_dataset.classes
print(f"Total images: {len(full_dataset)}")
print(f"Classes: {len(classes)}")

train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_ds, val_ds, test_ds = random_split(full_dataset, [train_size, val_size, test_size])
val_ds.dataset.transform = val_transforms
test_ds.dataset.transform = val_transforms

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

In [ ]:
# Cell 5 — Model
model = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print(f"Model ready — output classes: {NUM_CLASSES}")

In [ ]:
# Cell 6 — Training loop
with mlflow.start_run(run_name="efficientnet_b4_200species"):
    mlflow.log_params({
        "model": "efficientnet_b4",
        "num_classes": NUM_CLASSES,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "img_size": IMG_SIZE,
        "device": str(DEVICE)
    })

    for epoch in range(EPOCHS):
        model.train()
        train_loss, train_correct = 0, 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            train_correct += (outputs.argmax(1) == labels).sum().item()

        train_acc = train_correct / len(train_ds)

        model.eval()
        val_correct = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                val_correct += (outputs.argmax(1) == labels).sum().item()

        val_acc = val_correct / len(val_ds)
        scheduler.step()

        mlflow.log_metrics({
            "train_accuracy": train_acc,
            "val_accuracy": val_acc,
            "train_loss": train_loss / len(train_loader)
        }, step=epoch)

        print(f"Epoch {epoch+1}/{EPOCHS} | Train: {train_acc:.4f} | Val: {val_acc:.4f}")

    # Save model
    torch.save({
        'model_state_dict': model.state_dict(),
        'classes': classes,
        'num_classes': NUM_CLASSES
    }, MODEL_SAVE_PATH)
    mlflow.log_artifact(str(MODEL_SAVE_PATH))
    print(f"\nModel saved to {MODEL_SAVE_PATH}")

In [ ]:
# Cell 7 — Test evaluation
from sklearn.metrics import classification_report

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

test_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"Test accuracy: {test_acc:.4f}")
mlflow.log_metric("test_accuracy", test_acc)

print(classification_report(all_labels, all_preds, target_names=classes))